# PSMPY MODULE - please change data I am using Dummy data

In [5]:
import numpy as np
import pandas as pd
from psmpy import PsmPy
from scipy.spatial.distance import cdist

rng = np.random.default_rng(42)

# ======================= SETTINGS =======================
N_TEST = 20
N_POOL = 180
K = 3                        # controls wanted per test store
MATCH_ON = "mahalanobis"     # "mahalanobis" (traits directly) or "propensity" (PsmPy score)
CALIPER_Q = 0.05             # mahalanobis: keep pairs within the closest 5% of all test-control distances
CALIPER_SD = 0.3             # propensity: max logit gap = 0.3 x SD of the logit score

ID_COL = "store_id"
TREAT_COL = "is_test"

# Store traits used for matching (numeric only). No region code, no sales results.
features = ["avg_weekly_sales", "sq_ft", "avg_basket",
            "median_income", "competitors_5mi", "sales_growth_yoy"]

# ======================= 1. DUMMY DATA =======================
# Replace this block with: df = pd.read_csv("your_stores.csv")
n = N_TEST + N_POOL
is_test = np.r_[np.ones(N_TEST), np.zeros(N_POOL)].astype(int)
rng.shuffle(is_test)

sq_ft = rng.normal(75000, 15000, n).clip(35000, 120000)
income = rng.normal(70000, 18000, n).clip(30000, 150000)
sales = (sq_ft * 9 + income * 1.5 + rng.normal(0, 40000, n)).clip(300000, None)
sq_ft[is_test == 1] += 8000          # make test stores a bit bigger/richer
income[is_test == 1] += 7000
sales[is_test == 1] += 60000

df = pd.DataFrame({
    "store_id": [f"HEB_{i:04d}" for i in range(n)],
    "region": rng.integers(1, 41, n),            # kept in output, NOT used for matching
    "is_test": is_test,
    "avg_weekly_sales": sales,
    "sq_ft": sq_ft,
    "avg_basket": rng.normal(62, 9, n).clip(35, 95),
    "median_income": income,
    "competitors_5mi": rng.poisson(3, n),
    "sales_growth_yoy": rng.normal(0.03, 0.02, n),
})

# ======================= 2. PsmPy PROPENSITY SCORES =======================
# PsmPy only sees: ID + treatment flag + matching traits
psm_df = df[[ID_COL, TREAT_COL] + features].copy()
psm = PsmPy(psm_df, treatment=TREAT_COL, indx=ID_COL)
psm.logistic_ps(balance=True)
scores = psm.predicted_data.copy()

if ID_COL not in scores.columns:
    scores = scores.reset_index().rename(columns={"index": ID_COL})
if "propensity_logit" not in scores.columns:
    p = scores["propensity_score"].clip(1e-6, 1 - 1e-6)
    scores["propensity_logit"] = np.log(p / (1 - p))
scores = scores.set_index(ID_COL)

d = df.set_index(ID_COL)
d["propensity_score"] = scores.loc[d.index, "propensity_score"]
d["propensity_logit"] = scores.loc[d.index, "propensity_logit"]

test_ids = d.index[d[TREAT_COL] == 1]
ctrl_ids = d.index[d[TREAT_COL] == 0]

# ======================= 3. DISTANCE: every test store vs every control =======================
if MATCH_ON == "mahalanobis":
    X = d[features].values.astype(float)
    VI = np.linalg.pinv(np.cov(X, rowvar=False))          # handles correlated traits
    D = pd.DataFrame(cdist(d.loc[test_ids, features].values.astype(float),
                           d.loc[ctrl_ids, features].values.astype(float),
                           metric="mahalanobis", VI=VI),
                     index=test_ids, columns=ctrl_ids)
    cutoff = np.quantile(D.values, CALIPER_Q)
else:
    tl = d.loc[test_ids, "propensity_logit"].values
    cl = d.loc[ctrl_ids, "propensity_logit"].values
    D = pd.DataFrame(np.abs(tl[:, None] - cl[None, :]), index=test_ids, columns=ctrl_ids)
    cutoff = CALIPER_SD * d["propensity_logit"].std()

print(f"Match method: {MATCH_ON} | caliper cutoff: {cutoff:.4f}")

# ======================= 4. MATCHING: K nearest, caliper, controls reusable =======================
rows = []
dropped = []
for t_id in D.index:
    best = D.loc[t_id].sort_values().head(K)      # K nearest (control pool is never shrunk -> reuse)
    best = best[best <= cutoff]                   # caliper: poor matches dropped
    if len(best) == 0:
        dropped.append(t_id)
    for rank, (c_id, dist) in enumerate(best.items(), start=1):
        rows.append({"test_store": t_id, "control_store": c_id, "rank": rank, "distance": dist})

pairs = pd.DataFrame(rows, columns=["test_store", "control_store", "rank", "distance"])

# ======================= 5. EVALUATION =======================
print("\n" + "=" * 60)
print("CHECK 1: COVERAGE (did every test store find lookalikes?)")
print("=" * 60)
matched_n = pairs.test_store.nunique()
print(f"Test stores matched : {matched_n} / {N_TEST}")
print(f"Dropped test stores : {dropped if dropped else 'none'}")
print("Controls found per test store (count of test stores):")
print(pairs.groupby("test_store").size().value_counts().sort_index().to_string())
print(f"Unique control stores used: {pairs.control_store.nunique()}")
print("Most reused controls (times used):")
print(pairs.control_store.value_counts().head(5).to_string())

print("\n" + "=" * 60)
print("CHECK 2: BALANCE (are test and control groups alike?)")
print("SMD: below 0.1 = very good | 0.1-0.25 = acceptable | above 0.25 = poor")
print("=" * 60)
test_all = d.loc[test_ids]
ctrl_all = d.loc[ctrl_ids]
test_m = d.loc[pairs.test_store.unique()]
ctrl_m = d.loc[pairs.control_store]               # repeats allowed (reuse)

def smd(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

bal = []
for f in features:
    before, after = smd(test_all[f], ctrl_all[f]), smd(test_m[f], ctrl_m[f])
    status = "VERY GOOD" if abs(after) < 0.1 else "ACCEPTABLE" if abs(after) <= 0.25 else "POOR"
    bal.append({"trait": f, "SMD_before": round(before, 3),
                "SMD_after": round(after, 3), "status": status})
bal = pd.DataFrame(bal)
print(bal.to_string(index=False))

print("\n" + "=" * 60)
print("CHECK 3: PAIR-LEVEL (how close is each test store to its controls?)")
print("=" * 60)
trait_sd = d[features].std()                       # spread of each trait across all stores
gap = []
for t_id, g in pairs.groupby("test_store"):
    ctrl_mean = d.loc[g.control_store, features].mean()
    diff_sd = ((ctrl_mean - d.loc[t_id, features]) / trait_sd).abs()   # gap in SD units
    gap.append({"test_store": t_id, "n_controls": len(g),
                "avg_gap_sd": diff_sd.mean(), "worst_trait": diff_sd.idxmax(),
                "worst_gap_sd": diff_sd.max()})
gap = pd.DataFrame(gap).sort_values("avg_gap_sd")
print("Gap = difference between a test store and the average of its controls,")
print("in standard deviations of that trait (below 0.25 = close, above 0.5 = weak).")
print(f"Average gap across all pairs: {gap.avg_gap_sd.mean():.2f}")
print("Best 3 matched test stores:")
print(gap.head(3).round(2).to_string(index=False))
print("Weakest 3 matched test stores:")
print(gap.tail(3).round(2).to_string(index=False))
weak = (gap.avg_gap_sd > 0.5).sum()
print(f"Test stores with a weak match (avg gap > 0.5): {weak}")

print("\n" + "=" * 60)
print("OVERALL VERDICT")
print("=" * 60)
n_good = (bal.status == "VERY GOOD").sum()
n_poor = (bal.status == "POOR").sum()
print(f"Stores matched    : {matched_n}/{N_TEST}")
print(f"Traits very good  : {n_good}/{len(features)}  | poor: {n_poor}/{len(features)}")
print(f"Avg pair gap (SD) : {gap.avg_gap_sd.mean():.2f}  | weak pairs: {weak}")
if matched_n >= 0.8 * N_TEST and n_poor == 0:
    print("Verdict: GOOD match. Control stores are a fair lookalike group.")
elif n_poor > 0:
    print("Verdict: PARTIAL. Some traits still differ. Tighten CALIPER_Q or review the traits.")
else:
    print("Verdict: LOW COVERAGE. Many test stores dropped. Loosen CALIPER_Q or widen the control pool.")

# ======================= 6. OUTPUT (all columns) =======================
show_cols = [c for c in d.columns if c != TREAT_COL]
test_part = d.loc[pairs.test_store, show_cols].add_prefix("test_").reset_index(drop=True)
ctrl_part = d.loc[pairs.control_store, show_cols].add_prefix("ctrl_").reset_index(drop=True)
full_pairs = pd.concat([pairs.reset_index(drop=True), test_part, ctrl_part], axis=1)

print("\nMatched list (test store -> control stores):")
print(pairs.groupby("test_store")["control_store"].apply(list).to_string())

full_pairs.to_csv("matched_stores_all_columns.csv", index=False)
bal.to_csv("balance_check.csv", index=False)
print("\nSaved: matched_stores_all_columns.csv, balance_check.csv")

Match method: mahalanobis | caliper cutoff: 1.9283

CHECK 1: COVERAGE (did every test store find lookalikes?)
Test stores matched : 20 / 20
Dropped test stores : none
Controls found per test store (count of test stores):
1     1
2     1
3    18
Unique control stores used: 47
Most reused controls (times used):
control_store
HEB_0156    2
HEB_0177    2
HEB_0008    2
HEB_0036    2
HEB_0078    2

CHECK 2: BALANCE (are test and control groups alike?)
SMD: below 0.1 = very good | 0.1-0.25 = acceptable | above 0.25 = poor
           trait  SMD_before  SMD_after     status
avg_weekly_sales       0.611      0.221 ACCEPTABLE
           sq_ft       0.695      0.277       POOR
      avg_basket       0.242     -0.023  VERY GOOD
   median_income       0.425      0.046  VERY GOOD
 competitors_5mi       0.111     -0.001  VERY GOOD
sales_growth_yoy      -0.250     -0.060  VERY GOOD

CHECK 3: PAIR-LEVEL (how close is each test store to its controls?)
Gap = difference between a test store and the average

## Method 2 simple Nearby neighbour


In [6]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

# ======================= SETTINGS =======================
N_TEST = 20
N_POOL = 180
K = 3                          # controls wanted per test store (maximum)
MIN_CONTROLS = 2               # a test store needs at least this many controls to count as matched
MATCH_ON = "mahalanobis"       # "mahalanobis" (all traits directly) or "propensity" (sklearn score)
CALIPER_GRID = [0.10, 0.05, 0.03, 0.02, 0.01]   # caliper options to try (share of closest pairs kept)
MIN_COVERAGE = 0.80            # keep at least 80% of test stores
MAX_SMD = 0.25                 # no trait may be worse than this after matching

ID_COL, TREAT_COL = "store_id", "is_test"
features = ["avg_weekly_sales", "sq_ft", "avg_basket",
            "median_income", "competitors_5mi", "sales_growth_yoy"]

# ======================= 1. DUMMY DATA =======================
# Replace this block with: df = pd.read_csv("your_stores.csv")
n = N_TEST + N_POOL
is_test = np.r_[np.ones(N_TEST), np.zeros(N_POOL)].astype(int)
rng.shuffle(is_test)
sq_ft = rng.normal(75000, 15000, n).clip(35000, 120000)
income = rng.normal(70000, 18000, n).clip(30000, 150000)
sales = (sq_ft * 9 + income * 1.5 + rng.normal(0, 40000, n)).clip(300000, None)
sq_ft[is_test == 1] += 8000
income[is_test == 1] += 7000
sales[is_test == 1] += 60000
df = pd.DataFrame({
    "store_id": [f"HEB_{i:04d}" for i in range(n)],
    "region": rng.integers(1, 41, n),            # kept in output, NOT used for matching
    "is_test": is_test,
    "avg_weekly_sales": sales, "sq_ft": sq_ft,
    "avg_basket": rng.normal(62, 9, n).clip(35, 95),
    "median_income": income,
    "competitors_5mi": rng.poisson(3, n),
    "sales_growth_yoy": rng.normal(0.03, 0.02, n),
})

# ======================= 2. DISTANCE: every test store vs every control =======================
d = df.set_index(ID_COL)
test_ids = d.index[d[TREAT_COL] == 1]
ctrl_ids = d.index[d[TREAT_COL] == 0]

if MATCH_ON == "mahalanobis":
    X = d[features].values.astype(float)
    VI = np.linalg.pinv(np.cov(X, rowvar=False))          # handles correlated traits
    D = pd.DataFrame(cdist(d.loc[test_ids, features].values.astype(float),
                           d.loc[ctrl_ids, features].values.astype(float),
                           metric="mahalanobis", VI=VI),
                     index=test_ids, columns=ctrl_ids)
else:
    Xs = StandardScaler().fit_transform(d[features])
    p = LogisticRegression(C=1.0, max_iter=1000).fit(Xs, d[TREAT_COL]).predict_proba(Xs)[:, 1]
    d["propensity_score"] = p
    d["propensity_logit"] = np.log(p / (1 - p))
    tl = d.loc[test_ids, "propensity_logit"].values
    cl = d.loc[ctrl_ids, "propensity_logit"].values
    D = pd.DataFrame(np.abs(tl[:, None] - cl[None, :]), index=test_ids, columns=ctrl_ids)

# ======================= 3. TRY EACH CALIPER, SCORE THE RESULT =======================
trait_sd = d[features].std()
test_all, ctrl_all = d.loc[test_ids], d.loc[ctrl_ids]
smd = lambda a, b: (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

results, pairs_by_q, dropped_by_q = [], {}, {}
for q in CALIPER_GRID:
    cutoff = np.quantile(D.values, q)
    rows, dropped = [], []
    for t_id in D.index:
        best = D.loc[t_id].sort_values().head(K)      # K nearest; pool never shrinks -> reuse allowed
        best = best[best <= cutoff]                   # caliper: poor matches dropped
        if len(best) == 0:
            dropped.append(t_id)
        for rank, (c_id, dist) in enumerate(best.items(), start=1):
            rows.append({"test_store": t_id, "control_store": c_id, "rank": rank, "distance": dist})
    pairs = pd.DataFrame(rows, columns=["test_store", "control_store", "rank", "distance"])
    cnt = pairs.groupby("test_store").size()
    keep = cnt[cnt >= MIN_CONTROLS].index                 # need at least MIN_CONTROLS good controls
    dropped = [t for t in D.index if t not in keep]
    pairs = pairs[pairs.test_store.isin(keep)]
    pairs_by_q[q], dropped_by_q[q] = pairs, dropped
    if len(pairs) == 0:
        results.append({"caliper_q": q, "matched": 0, "full_3": 0, "avg_smd": np.nan,
                        "max_smd": np.nan, "weak_pairs": 0})
        continue

    test_m = d.loc[pairs.test_store.unique()]
    ctrl_m = d.loc[pairs.control_store]
    smds = pd.Series({f: smd(test_m[f], ctrl_m[f]) for f in features}).abs()
    weak = 0
    for t_id, g in pairs.groupby("test_store"):
        gap = ((d.loc[g.control_store, features].mean() - d.loc[t_id, features]) / trait_sd).abs().mean()
        weak += gap > 0.5
    results.append({"caliper_q": q, "matched": pairs.test_store.nunique(),
                    "full_3": int((pairs.groupby("test_store").size() == K).sum()),
                    "avg_smd": smds.mean(), "max_smd": smds.max(), "weak_pairs": int(weak)})

res = pd.DataFrame(results)
print("CALIPER COMPARISON (smaller caliper_q = stricter)")
print(res.round(3).to_string(index=False))

# ======================= 4. PICK THE BEST CALIPER =======================
ok = res[(res.matched >= MIN_COVERAGE * N_TEST) & (res.max_smd <= MAX_SMD)]
if len(ok) > 0:
    best_q = ok.sort_values(["avg_smd", "matched"], ascending=[True, False]).iloc[0].caliper_q
    why = "meets coverage and balance targets, lowest average SMD"
else:
    cov = res[res.matched >= MIN_COVERAGE * N_TEST]
    pool_ = cov if len(cov) > 0 else res
    best_q = pool_.sort_values("max_smd").iloc[0].caliper_q
    why = "no setting met every target; this is the closest"
print(f"\nChosen caliper_q = {best_q}  ({why})")

pairs, dropped = pairs_by_q[best_q], dropped_by_q[best_q]

# ======================= 5. EVALUATION OF THE CHOSEN MATCH =======================
matched_n = pairs.test_store.nunique()
print("\n--- COVERAGE ---")
print(f"Test stores matched: {matched_n} / {N_TEST}   | dropped: {dropped if dropped else 'none'}")
print("Controls per test store (count of test stores):")
print(pairs.groupby("test_store").size().value_counts().sort_index().to_string())
print(f"Unique controls used: {pairs.control_store.nunique()}  | most reused:")
print(pairs.control_store.value_counts().head(3).to_string())

print("\n--- BALANCE (SMD: <0.1 very good | 0.1-0.25 acceptable | >0.25 poor) ---")
test_m, ctrl_m = d.loc[pairs.test_store.unique()], d.loc[pairs.control_store]
bal = pd.DataFrame([{"trait": f, "SMD_before": round(smd(test_all[f], ctrl_all[f]), 3),
                     "SMD_after": round(smd(test_m[f], ctrl_m[f]), 3)} for f in features])
bal["status"] = np.where(bal.SMD_after.abs() < 0.1, "VERY GOOD",
                 np.where(bal.SMD_after.abs() <= 0.25, "ACCEPTABLE", "POOR"))
print(bal.to_string(index=False))

print("\n--- PAIR LEVEL (gap in SDs: <0.25 close | >0.5 weak) ---")
gap = []
for t_id, g in pairs.groupby("test_store"):
    diff = ((d.loc[g.control_store, features].mean() - d.loc[t_id, features]) / trait_sd).abs()
    gap.append({"test_store": t_id, "n_controls": len(g), "avg_gap_sd": diff.mean(),
                "worst_trait": diff.idxmax(), "worst_gap_sd": diff.max()})
gap = pd.DataFrame(gap).sort_values("avg_gap_sd")
print(f"Average gap: {gap.avg_gap_sd.mean():.2f}")
print("Weakest 3:"); print(gap.tail(3).round(2).to_string(index=False))

print("\n--- VERDICT ---")
n_poor = (bal.status == "POOR").sum()
if matched_n >= MIN_COVERAGE * N_TEST and n_poor == 0:
    print("GOOD: control stores are a fair lookalike group.")
elif n_poor > 0:
    print("PARTIAL: some traits still differ. Review the traits or widen the control pool.")
else:
    print("LOW COVERAGE: many test stores dropped. Loosen the caliper or widen the control pool.")

# ======================= 6. OUTPUT (all columns) =======================
show_cols = [c for c in d.columns if c != TREAT_COL]
full_pairs = pd.concat([pairs.reset_index(drop=True),
                        d.loc[pairs.test_store, show_cols].add_prefix("test_").reset_index(drop=True),
                        d.loc[pairs.control_store, show_cols].add_prefix("ctrl_").reset_index(drop=True)], axis=1)
print("\nMatched list (test store -> controls):")
print(pairs.groupby("test_store")["control_store"].apply(list).to_string())
full_pairs.to_csv("matched_stores_all_columns.csv", index=False)
bal.to_csv("balance_check.csv", index=False)
res.to_csv("caliper_comparison.csv", index=False)
print("\nSaved: matched_stores_all_columns.csv, balance_check.csv, caliper_comparison.csv")

CALIPER COMPARISON (smaller caliper_q = stricter)
 caliper_q  matched  full_3  avg_smd  max_smd  weak_pairs
      0.10       20      19    0.097    0.254           3
      0.05       19      18    0.081    0.213           3
      0.03       18      15    0.082    0.208           2
      0.02       14      12    0.087    0.173           0
      0.01       11       6    0.094    0.217           0

Chosen caliper_q = 0.05  (meets coverage and balance targets, lowest average SMD)

--- COVERAGE ---
Test stores matched: 19 / 20   | dropped: ['HEB_0087']
Controls per test store (count of test stores):
2     1
3    18
Unique controls used: 46  | most reused:
control_store
HEB_0156    2
HEB_0177    2
HEB_0036    2

--- BALANCE (SMD: <0.1 very good | 0.1-0.25 acceptable | >0.25 poor) ---
           trait  SMD_before  SMD_after     status
avg_weekly_sales       0.611      0.159 ACCEPTABLE
           sq_ft       0.695      0.213 ACCEPTABLE
      avg_basket       0.242      0.005  VERY GOOD
   medi